In [3]:
import numpy as np
import pandas as pd
import torch
import os
import random
import pyarrow as pa
from scipy import sparse
from pathlib import Path

In [1]:
# =========================
# Step 1 · 环境与依赖初始化（PyTorch 多 GPU 基座）
# =========================

# ---- 标准库 ----
import os
import re
import math
import json
import time
import random
from pathlib import Path
from typing import Optional, Tuple, List

# ---- 科学计算 / 数据 IO ----
import numpy as np
import pandas as pd

# Parquet 要求：使用 fastparquet
# pip install fastparquet
# （若需要更高性能也可选 pyarrow，但本项目统一 fastparquet）
PARQUET_ENGINE = "fastparquet"

# 稀疏矩阵（用于与已有管线兼容的三元组落盘；随后将逐步转换为 PyTorch 稀疏）
from scipy import sparse

# ---- PyTorch 栈 ----
import torch
import torch.nn as nn
import torch.nn.functional as F

# 可选：分布式 / 多 GPU。Notebook 推荐 DataParallel；脚本/集群可用 DDP（torchrun）
import torch.distributed as dist

# 提升 matmul 的 float32 精度/吞吐（PyTorch 2.0+）
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# cudnn benchmark 有益于卷积型工作负载；对本项目影响不大，但开启通常无害
torch.backends.cudnn.benchmark = True

# ---- 近邻搜索（后续 Step 7+ 用）----
# 推荐 GPU 版：pip install faiss-gpu==1.7.4
# CPU 版：pip install faiss-cpu==1.7.4
try:
    import faiss  # noqa
    _FAISS_AVAILABLE = True
except Exception:
    _FAISS_AVAILABLE = False

# ---- 通用路径 ----
TMP_DIR = Path("./tmp")  # 统一的中间件目录（你要求）
TMP_DIR.mkdir(exist_ok=True)

# ---- 随机种子 ----
def set_seed(seed: int = 2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(2025)

# ---- GPU/分布式环境工具 ----
def gpu_summary():
    n = torch.cuda.device_count()
    if n == 0:
        print("[GPU] No CUDA device found. Running on CPU.")
        return
    print(f"[GPU] CUDA devices: {n}")
    for i in range(n):
        prop = torch.cuda.get_device_properties(i)
        print(f"  - GPU{i}: {prop.name}, {prop.total_memory/1024**3:.1f} GB, SMs={prop.multi_processor_count}")

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def use_all_gpus_for_module(module: nn.Module) -> nn.Module:
    """
    Notebook/单进程场景：用 DataParallel 覆盖所有 GPU。
    如使用 torchrun/Slurm 多进程 DDP，请在外部初始化后改用 DDP 包装。
    """
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        return nn.DataParallel(module)  # 简便多GPU
    return module

def maybe_init_ddp(backend: str = "nccl"):
    """
    仅当你用 torchrun 启动时才会生效；Notebook 默认不做 DDP 初始化。
    export MASTER_ADDR=... MASTER_PORT=... WORLD_SIZE=... RANK=...
    torchrun --nproc_per_node=NUM_GPUS your_script.py
    """
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        if not dist.is_initialized():
            dist.init_process_group(backend=backend)
            local_rank = int(os.environ.get("LOCAL_RANK", 0))
            torch.cuda.set_device(local_rank)
            print(f"[DDP] Initialized. rank={dist.get_rank()}/{dist.get_world_size()}, local_rank={local_rank}")
    else:
        # Notebook 情况：不启 DDP
        pass

gpu_summary()
print(f"[Env] TMP_DIR = {TMP_DIR.resolve()}")

# ---- Parquet / 稀疏 CSR 三元组 读写工具（与历史管线兼容） ----
def save_parquet_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, engine=PARQUET_ENGINE, index=False)

def load_parquet_df(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path, engine=PARQUET_ENGINE)

def save_csr_as_triplets(mat: sparse.csr_matrix, path: Path, dtype=np.float32):
    coo = mat.tocoo()
    pdf = pd.DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(dtype),
    })
    save_parquet_df(pdf, path)

def load_csr_from_triplets(path: Path, shape: Tuple[int, int], dtype=np.float32) -> sparse.csr_matrix:
    pdf = load_parquet_df(path)
    coo = sparse.coo_matrix((pdf["val"].astype(dtype), (pdf["row"], pdf["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

# ---- Torch 稀疏工具（后续步骤会逐步改为 torch.sparse 格式进行 GPU 计算）----
def csr_to_torch_coo(mat: sparse.csr_matrix, device: torch.device = None, dtype: torch.dtype = torch.float32):
    mat = mat.tocoo()
    indices = torch.tensor(np.vstack([mat.row, mat.col]), dtype=torch.long, device=device)
    values  = torch.tensor(mat.data, dtype=dtype, device=device)
    shape   = torch.Size(mat.shape)
    return torch.sparse_coo_tensor(indices, values, shape, device=device, dtype=dtype).coalesce()

def torch_row_l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    对 dense 或稀疏的行做 L2 归一（对稀疏COO：只缩放 values）。
    这里先提供 dense 版本；后续在具体步骤对稀疏做定制归一。
    """
    if x.is_sparse:
        # 稀疏归一在后续针对性实现；此处仅放占位说明
        raise NotImplementedError("Sparse L2 normalize will be implemented in the graph-specific steps.")
    norm = torch.linalg.norm(x, dim=1, keepdim=True).clamp_min(eps)
    return x / norm

print("[Step 1] PyTorch 多 GPU 环境初始化完成。后续步骤将沿用以上工具与路径。")


[GPU] CUDA devices: 2
  - GPU0: NVIDIA RTX 6000 Ada Generation, 47.5 GB, SMs=142
  - GPU1: NVIDIA RTX 6000 Ada Generation, 47.5 GB, SMs=142
[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[Step 1] PyTorch 多 GPU 环境初始化完成。后续步骤将沿用以上工具与路径。
